In [ ]:
import pandas as pd
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
import requests 
import os
import numpy as np
import pyarrow
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path



In [ ]:
# Definition des dossiers
git_folder = "Patricia-Promise-Immo"
folder_entries = "data"
base_dir = Path().resolve().parents[1]
data_dir = base_dir/git_folder/folder_entries
if not data_dir.exists():
    data_dir.mkdir(parents=True)
print(f"Data directory is set to : {data_dir}")

Data directory is set to : D:\ProjectFolderDevAI_2025-2026\Immo_project\Patricia-Promise-Immo\data


In [ ]:
# Liste des fichiers à télécharger par année via leurs URLs
DOWNLOADs_ = [
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234902/valeursfoncieres-2025-s1.txt.zip',
        'year' : '2025'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234857/valeursfoncieres-2024.txt.zip',
        'year' : '2024'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234851/valeursfoncieres-2023.txt.zip',
        'year' : '2023'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234844/valeursfoncieres-2022.txt.zip',
        'year' : '2022'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234836/valeursfoncieres-2021.txt.zip',
        'year' : '2021'
    },
    {
        'url' : 'https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20251018-234831/valeursfoncieres-2020-s2.txt.zip',
        'year' : '2020'
    }
    
    ]

# Téléchargement des fichiers par année
for immo_file in DOWNLOADs_ :
    response = requests.get(immo_file['url'], stream=True) #Stream = True afin d'éviter de tout charger en mémoire
    
    # Création du dossier 'raw' s'il n'existe pas pour stocker les fichiers téléchargés
    raw_dir = data_dir/"raw"
    if not raw_dir.exists():
        raw_dir.mkdir(parents=True)
        
    # Enregistrement du fichier par itérations
    with open(f"{raw_dir}/immo_entries_{immo_file['year']}.txt.zip", 'wb') as file:
        for chunk in response.iter_content(chunk_size=10_000):
            file.write(chunk)

print("Téléchargement terminé.")

In [ ]:
# Liste des fichiers téléchargés
all_immo_files = os.listdir(raw_dir)
print("Fichiers téléchargés :")
for f in all_immo_files:
    print(f" - {f}")    

In [ ]:
# Exploration des fichiers téléchargés et conversion en DataFrame Pandas
for immo_file in all_immo_files:
    df = pd.read_csv(f"{raw_dir}/{immo_file}", nrows=100, on_bad_lines='skip', low_memory=False, sep='|', decimal=',', dtype={'Code postal':str, 'Valeur fonciere':np.float64})
    print(df.columns.to_list())


In [6]:
"""
    Objectif: enregistrement des csv en 1 seul par chunks pour gérer efficacement la mémoire de la machine
    fichier final: all_immo_entries.csv dans un dossier 'interim'

"""
# Initialisation de la variable pour gérer l'écriture du header
first_file = True

# Création du dossier 'interim' s'il n'existe pas pour stocker les fichiers téléchargés
interim_dir = data_dir/"interim"
if not interim_dir.exists():
    interim_dir.mkdir(parents=True)

# Chemin du fichier de sortie
output_path = interim_dir/"all_immo_entries.csv"

for immo_file in all_immo_files:
    for index, chunk in enumerate(pd.read_csv(f'{raw_dir}/{immo_file}', on_bad_lines='skip', low_memory=False, sep='|', chunksize=10_000, decimal=',', dtype={'Code postal':str, 'Valeur fonciere':np.float64})):
        df.to_csv(f"{output_path}", mode = 'w' if first_file else 'a', header=first_file)
        first_file = False
    print(f'"{immo_file}" : DONE')

In [ ]:
parquet_path= interim_dir/"all_immo_entries.parquet"
print(parquet_path)

writer = None

for index, chunk in enumerate(pd.read_csv(f"{output_path}", chunksize=10_000)):
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    
    # Initialise le writer une seule fois avec le schéma du premier chunk
    if writer is None:
        writer = pq.ParquetWriter(parquet_path, schema=table.schema, compression="snappy")
    
    # Écrit le chunk courant
    writer.write_table(table)

# Ferme le writer à la fin
if writer is not None:
    writer.close()



In [ ]:
df = pd.read_parquet(parquet_path)

print("Parquet file info:")
df.head(20)
df.info()


In [5]:
df.shape

(201300, 44)

tests

In [24]:
import pandas as pd

In [25]:
df = pd.read_parquet('../outputs/features_immo_data.parquet')

In [31]:
DF2 = pd.read_parquet('../outputs/all_immo_no_empty.parquet')
print(DF2.shape)
DF2

(201300, 29)


,Unnamed: 0,No disposition,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Type de voie,Code voie,Voie,Code postal,Commune,Code departement,Code commune,Prefixe de section,Section,No plan,1er lot,Surface Carrez du 1er lot,2eme lot,Surface Carrez du 2eme lot,Nombre de lots,Code type local,Type local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain
0,0,1,07/01/2025,Vente,468000.0,NaN,None,None,B078,FARGES,1550,FARGES,1,158,NaN,B,815,NaN,NaN,NaN,NaN,0,NaN,None,NaN,NaN,J,None,78.0
1,1,1,07/01/2025,Vente,468000.0,454.0,None,RUE,0090,DE LA REPUBLIQUE,1550,FARGES,1,158,NaN,B,910,NaN,NaN,NaN,NaN,0,1.0,Maison,111.0,5.0,S,None,133.0
2,2,1,07/01/2025,Vente,468000.0,454.0,None,RUE,0090,DE LA REPUBLIQUE,1550,FARGES,1,158,NaN,B,910,NaN,NaN,NaN,NaN,0,3.0,Dépendance,0.0,0.0,S,None,133.0
3,3,1,06/01/2025,Vente,180000.0,NaN,None,None,B158,LE VILLAGE,1200,MONTANGES,1,257,NaN,AC,334,NaN,NaN,NaN,NaN,0,NaN,None,NaN,NaN,S,None,46.0
4,4,1,06/01/2025,Vente,180000.0,NaN,None,None,B158,LE VILLAGE,1200,MONTANGES,1,257,NaN,AC,338,NaN,NaN,NaN,NaN,0,NaN,None,NaN,NaN,J,None,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201295,95,1,07/01/2025,Vente,295000.0,NaN,None,None,B010,LA CURTERIE,1300,MAGNIEU,1,227,NaN,B,1847,NaN,NaN,NaN,NaN,0,NaN,None,NaN,NaN,AB,None,402.0
201296,96,1,07/01/2025,Vente,295000.0,NaN,None,None,B010,LA CURTERIE,1300,MAGNIEU,1,227,NaN,B,1848,NaN,NaN,NaN,NaN,0,NaN,None,NaN,NaN,AB,None,4.0
201297,97,1,06/01/2025,Vente,81500.0,NaN,None,None,B036,VILLAGE DE CHARNOZ,1800,CHARNOZ-SUR-AIN,1,88,NaN,B,396,NaN,NaN,NaN,NaN,0,NaN,None,NaN,NaN,J,None,261.0
201298,98,1,14/01/2025,Vente,78000.0,110.0,None,RUE,0443,DU PRE PAQUIER,1750,SAINT-LAURENT-SUR-SAONE,1,370,NaN,A,680,49.0,48.62,NaN,NaN,1,2.0,Appartement,48.0,2.0,None,None,NaN


In [ ]:
print(df.shape)
df

(201300, 17)


,No disposition,Valeur fonciere,Code postal,Code departement,Code commune,Prefixe de section,No plan,1er lot,Surface Carrez du 1er lot,2eme lot,Surface Carrez du 2eme lot,Nombre de lots,Code type local,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain
0,1,468000.0,1550,1,158,0.0,815,0.0,0.00,0.0,0.0,0,0.0,0.0,0.0,0.0,78.0
1,1,468000.0,1550,1,158,0.0,910,0.0,0.00,0.0,0.0,0,1.0,2.0,111.0,5.0,133.0
2,1,468000.0,1550,1,158,0.0,910,0.0,0.00,0.0,0.0,0,3.0,1.0,0.0,0.0,133.0
3,1,180000.0,1200,1,257,0.0,334,0.0,0.00,0.0,0.0,0,0.0,0.0,0.0,0.0,46.0
4,1,180000.0,1200,1,257,0.0,338,0.0,0.00,0.0,0.0,0,0.0,0.0,0.0,0.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201295,1,295000.0,1300,1,227,0.0,1847,0.0,0.00,0.0,0.0,0,0.0,0.0,0.0,0.0,402.0
201296,1,295000.0,1300,1,227,0.0,1848,0.0,0.00,0.0,0.0,0,0.0,0.0,0.0,0.0,4.0
201297,1,81500.0,1800,1,88,0.0,396,0.0,0.00,0.0,0.0,0,0.0,0.0,0.0,0.0,261.0
201298,1,78000.0,1750,1,370,0.0,680,49.0,48.62,0.0,0.0,1,2.0,3.0,48.0,2.0,0.0


In [29]:
correlations = df.corr(method="spearman")['Valeur fonciere']
correlations_sorted = correlations.abs().sort_values(ascending=False)
# Affichage propre avec les signes
print("Corrélation de Pearson avec 'prix' (triée par valeur absolue) :\n")
print(correlations.loc[correlations_sorted.index])

Corrélation de Pearson avec 'prix' (triée par valeur absolue) :

Valeur fonciere               1.000000
Code commune                 -0.313364
Code type local               0.280461
Type local                    0.223691
Nombre de lots               -0.205286
1er lot                      -0.197661
Prefixe de section           -0.172588
Surface reelle bati           0.170966
Nombre pieces principales     0.165950
Surface terrain               0.146394
No plan                       0.141735
Surface Carrez du 2eme lot   -0.131332
Code postal                   0.107041
Surface Carrez du 1er lot    -0.077876
No disposition                0.070161
2eme lot                     -0.034161
Code departement                   NaN
Name: Valeur fonciere, dtype: float64
